In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [207]:
path = 'cleaned_dataset/'

In [29]:
def parse_datetime(array_str):
    vals = np.fromstring(array_str.strip('[]'), sep=' ')
    return datetime(
        int(vals[0]), int(vals[1]), int(vals[2]),
        int(vals[3]), int(vals[4]),
        int(vals[5]), int((vals[5] % 1) * 1_000_000)
    )

In [225]:
def read_clean_file(filepath):
    """
    input: filepath to the 'cleaned_dataset' folder
    ex: 'user/downloads/cleaned_dataset'
    """
    filepath += '/metadata.csv'
    df = pd.read_csv(filepath)
    df['start_time'] = df['start_time'].apply(parse_datetime)

    if 'Capacity' in df.columns:
        df['Capacity'] = pd.to_numeric(df['Capacity'], errors='coerce')

    return df

In [243]:
def extract_cycle_features(filepath, cycle):
    """
    Input: 
        filepath to the 'cleaned_dataset' folder
        df_cycle = a single discharge CSV filename (time series)
    Output: dictionary of aggregated features for ML
    """
    filepath += '/data/' + cycle
    df_cycle = pd.read_csv(filepath)
    
    features = {}
    features['mean_voltage'] = df_cycle['Voltage_measured'].mean()
    features['max_voltage'] = df_cycle['Voltage_measured'].max()
    features['min_voltage'] = df_cycle['Voltage_measured'].min()
    
    features['mean_current'] = df_cycle['Current_measured'].mean()
    features['max_current'] = df_cycle['Current_measured'].max()
    
    features['mean_temperature'] = df_cycle['Temperature_measured'].mean()
    features['max_temperature'] = df_cycle['Temperature_measured'].max()
    
    features['discharge_time'] = df_cycle['Time'].max() - df_cycle['Time'].min()

    #Internal resistance estimate R = (VOC - V_load) / I

    #dataset has a tendency for discharges to start with really low current
    threshold = 0.1 
    valid_rows = df_cycle[np.abs(df_cycle['Current_load']) > threshold]
    
    if len(valid_rows) > 0:
        i_start = valid_rows.index[0]
    else:
        # fallback if dataset is weird
        i_start = df_cycle.index[5]
    
    I0 = abs(df_cycle.loc[i_start, 'Current_measured'])
    V0 = df_cycle.loc[i_start, 'Voltage_measured']
    
    # NASA method assumption for open-circuit voltage
    VOC = V0 + 0.04  #empirical offset used in PHM08 papers
    #internal resistance
    features['r_internal'] = (VOC - V0) / I0 if I0 > 0 else np.nan

    #mean slope dV/dt
    dVdt = np.gradient(df_cycle['Voltage_measured'], df_cycle['Time'])
    features['mean_dvdt'] = np.mean(dVdt)

    #Delivered capacity Ah = integral(I / 3600)dt
    capacity_As = np.trapz(
                abs(df_cycle['Current_measured']),
                df_cycle['Time']
            )
    features['capacity_Ah'] = capacity_As / 3600

    voltage_drop = (
                df_cycle['Voltage_measured'].iloc[0] -
                df_cycle['Voltage_measured'].iloc[-1]
            )
    features['voltage_drop'] = voltage_drop
    
    
    return features

In [245]:
def get_discharges(filepath, df):
    """
    input: 
        filepath to the 'cleaned_dataset' folder
        main df
    output: df with only discharges and added cycles and cycle features
    """
    df_discharges = df[df['type'] == 'discharge'][['start_time', 'ambient_temperature', 'battery_id', 'uid', 'filename', 'Capacity']].copy()
    df_discharges['cycle_number'] = df_discharges.groupby('battery_id').cumcount() + 1
    
    mean_voltage = []
    max_voltage = []
    min_voltage = []
    
    mean_current = []
    max_current = []
    
    mean_temp = []
    max_temp = []
    discharge_time = []
    
    for _, row in df_discharges.iterrows():
        try:
            file = row['filename']
            feats = extract_cycle_features(filepath, file)
            mean_voltage.append(feats['mean_voltage'])
            max_voltage.append(feats['max_voltage'])
            min_voltage.append(feats['min_voltage'])
            mean_current.append(feats['mean_current'])
            max_current.append(feats['max_current'])
            mean_temp.append(feats['mean_temperature'])
            max_temp.append(feats['max_temperature'])
            discharge_time.append(feats['discharge_time'])
        except Exception as e:
            print(f'error: {e}')
            mean_voltage.append(None)
            max_voltage.append(None)
            min_voltage.append(None)
            mean_current.append(None)
            max_current.append(None)
            mean_temp.append(None)
            max_temp.append(None)
            discharge_time.append(None)
    df_discharges['mean_voltage'] = mean_voltage
    df_discharges['max_voltage'] = max_voltage
    df_discharges['min_voltage'] = min_voltage
    df_discharges['mean_current'] = mean_current
    df_discharges['max_current'] = max_current
    df_discharges['mean_temperature'] = mean_temp
    df_discharges['max_temperature'] = max_temp
    df_discharges['discharge_time'] = discharge_time

    df_discharges = df_discharges.dropna()


    return df_discharges
    
            
    
    

In [247]:
def get_discharges_phyiscs(filepath, df):
    """
    input: 
        filepath to the 'cleaned_dataset' folder
        main df
    output: df with only discharges and added cycles and cycle features
    """
    df_discharges = df[df['type'] == 'discharge'][['start_time', 'ambient_temperature', 'battery_id', 'uid', 'filename', 'Capacity']].copy()
    df_discharges['cycle_number'] = df_discharges.groupby('battery_id').cumcount() + 1
    
    mean_voltage = []
    max_voltage = []
    min_voltage = []
    
    mean_current = []
    max_current = []
    
    mean_temp = []
    max_temp = []
    discharge_time = []

    R_internal = []
    mean_dVdt = []
    capacity_Ah = []
    capacity_ratio = []
    voltage_drop = []
    
    for _, row in df_discharges.iterrows():
        try:
            file = row['filename']
            feats = extract_cycle_features(filepath, file)
            mean_voltage.append(feats['mean_voltage'])
            max_voltage.append(feats['max_voltage'])
            min_voltage.append(feats['min_voltage'])
            mean_current.append(feats['mean_current'])
            max_current.append(feats['max_current'])
            mean_temp.append(feats['mean_temperature'])
            max_temp.append(feats['max_temperature'])
            discharge_time.append(feats['discharge_time'])
            R_internal.append(feats['r_internal'])
            mean_dVdt.append(feats['mean_dvdt'])
            capacity_Ah.append(feats['capacity_Ah'])
            voltage_drop.append(feats['voltage_drop'])

            capacity_ratio.append(feats['capacity_Ah'] / row['Capacity'])
        except Exception as e:
            print(f'error: {e}')
            mean_voltage.append(np.nan)
            max_voltage.append(np.nan)
            min_voltage.append(np.nan)
            mean_current.append(np.nan)
            max_current.append(np.nan)
            mean_temp.append(np.nan)
            max_temp.append(np.nan)
            discharge_time.append(np.nan)
            R_internal.append(np.nan)
            mean_dVdt.append(np.nan)
            capacity_Ah.append(np.nan)
            capacity_ratio.append(np.nan)
            voltage_drop.append(np.nan)
    df_discharges['mean_voltage'] = mean_voltage
    df_discharges['max_voltage'] = max_voltage
    df_discharges['min_voltage'] = min_voltage
    df_discharges['mean_current'] = mean_current
    df_discharges['max_current'] = max_current
    df_discharges['mean_temperature'] = mean_temp
    df_discharges['max_temperature'] = max_temp
    df_discharges['discharge_time'] = discharge_time
    df_discharges['r_internal'] = R_internal
    df_discharges['mean_dvdt'] = mean_dVdt
    df_discharges['capacity_Ah'] = capacity_Ah
    df_discharges['capacity_ratio'] = capacity_ratio
    df_discharges['voltage_drop'] = voltage_drop

    df_discharges = df_discharges.dropna()


    return df_discharges

In [263]:
df = read_clean_file(path)
df[0:185]

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:35.093000,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,2010-07-21 16:53:45.968000,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,2010-07-21 17:25:40.670999,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:05.000000,24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,2010-07-21 21:02:56.984000,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
180,discharge,2010-08-17 09:52:05.375000,4,B0047,180,181,00181.csv,1.1567087516841796,NaN,NaN
181,impedance,2010-08-17 11:12:01.125000,24,B0047,181,182,00182.csv,NaN,0.07436360239266564,0.23408631844470174
182,charge,2010-08-17 11:43:58.265000,4,B0047,182,183,00183.csv,NaN,NaN,NaN
183,impedance,2010-08-17 14:49:38.670999,24,B0047,183,184,00184.csv,NaN,0.07300469972093052,0.22171366635291487


In [239]:
df_discharges = get_discharges(path, df)
df_discharges

,start_time,ambient_temperature,battery_id,uid,filename,Capacity,cycle_number,mean_voltage,max_voltage,min_voltage,mean_current,max_current,mean_temperature,max_temperature,discharge_time
0,2010-07-21 15:00:35.093000,4,B0047,1,00001.csv,1.6743047446975208,1,3.475266,4.246764,2.470612,-0.952767,0.000252,8.272423,12.376816,6436.141
4,2010-07-21 21:02:56.984000,4,B0047,5,00005.csv,1.5243662105099023,2,3.476559,4.186636,2.477662,-0.983889,-0.001536,8.210715,11.314903,5650.265
6,2010-07-22 01:40:06.217999,4,B0047,7,00007.csv,1.5080762969973425,3,3.470767,4.199923,2.470710,-0.983889,-0.000746,7.954455,11.624528,5590.907
8,2010-07-22 06:16:21.780999,4,B0047,9,00009.csv,1.4835577960067696,4,3.467551,4.199569,2.465458,-0.978947,0.000303,7.985865,11.092924,5543.610
10,2010-07-22 10:51:48.203000,4,B0047,11,00011.csv,1.4671391666146525,5,3.462839,4.199397,2.465765,-0.976262,0.000422,8.009427,11.020979,5499.046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7553,2010-09-29 19:50:59.780999,4,B0055,7554,07554.csv,1.0285269676595319,98,3.370549,4.206432,2.495933,-1.676154,0.001937,10.889278,15.515116,2345.000
7555,2010-09-29 23:33:00.890000,4,B0055,7556,07556.csv,0.9816844358987022,99,3.358032,4.203481,2.493974,-1.656642,0.001502,11.125862,17.410883,2363.047
7557,2010-09-30 03:15:20.437000,4,B0055,7558,07558.csv,1.0127121434171131,100,3.330593,4.203063,2.494585,-1.736640,0.002544,11.180258,17.541633,2316.687
7561,2010-09-30 08:08:36.328000,4,B0055,7562,07562.csv,1.0201379996149256,101,3.345081,4.197674,2.497096,-1.713216,0.003379,11.102810,16.684126,2322.000


In [229]:
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:35.093000,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,2010-07-21 16:53:45.968000,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,2010-07-21 17:25:40.670999,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:05.000000,24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,2010-07-21 21:02:56.984000,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:45.045999,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,2010-09-30 08:08:36.328000,4,B0055,248,7562,07562.csv,1.0201379996149256,NaN,NaN
7562,charge,2010-09-30 08:48:54.250000,4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,2010-09-30 11:50:17.687000,4,B0055,250,7564,07564.csv,0.9907591663373165,NaN,NaN


In [104]:
BATTERIES = sorted(df['battery_id'].value_counts().index.tolist())
len(BATTERIES)

34

In [106]:
df[df['battery_id'] == 'B0007']['type'].value_counts()

type
impedance    278
charge       170
discharge    168
Name: count, dtype: int64

In [65]:
df[(df['battery_id'] == 'B0007') & (df['type'] == 'discharge')]

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
5737,discharge,2008-04-02 15:25:41.593000,24,B0007,1,5738,05738.csv,1.89105229539079,NaN,NaN
5739,discharge,2008-04-02 19:43:48.405999,24,B0007,3,5740,05740.csv,1.880637027686859,NaN,NaN
5741,discharge,2008-04-03 00:01:06.687000,24,B0007,5,5742,05742.csv,1.8806626717011388,NaN,NaN
5743,discharge,2008-04-03 04:16:37.375000,24,B0007,7,5744,05744.csv,1.8807709009833444,NaN,NaN
5745,discharge,2008-04-03 08:33:25.702999,24,B0007,9,5746,05746.csv,1.8794508728285058,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
6335,discharge,2008-05-26 10:44:38.468000,24,B0007,599,6336,06336.csv,1.4061714290398801,NaN,NaN
6339,discharge,2008-05-26 15:30:43.968000,24,B0007,603,6340,06340.csv,1.4063358479692922,NaN,NaN
6343,discharge,2008-05-26 20:21:04.921000,24,B0007,607,6344,06344.csv,1.4004552399066514,NaN,NaN
6347,discharge,2008-05-27 15:52:41.359000,24,B0007,611,6348,06348.csv,1.4217865046055154,NaN,NaN


In [201]:
df_discharges = df[df['type'] == 'discharge'][['start_time', 'ambient_temperature', 'battery_id', 'uid', 'filename', 'Capacity']]
df_discharges['cycle_number'] = df_discharges.groupby('battery_id').cumcount() + 1

"""
tester code that shows all cycle numbers are labeled correctly, each battery is only 'seen' once and has cycle number 1
seen = []
for i in range(1, len(df_discharges)):
    if df_discharges.iloc[i]['battery_id'] != df_discharges.iloc[i-1]['battery_id']:
        print(i, df_discharges.iloc[i])
        seen.append(df_discharges.iloc[i]['battery_id'])
print(seen)
"""

"\ntester code that shows all cycle numbers are labeled correctly, each battery is only 'seen' once and has cycle number 1\nseen = []\nfor i in range(1, len(df_discharges)):\n    if df_discharges.iloc[i]['battery_id'] != df_discharges.iloc[i-1]['battery_id']:\n        print(i, df_discharges.iloc[i])\n        seen.append(df_discharges.iloc[i]['battery_id'])\nprint(seen)\n"

In [217]:
first_file = df_discharges.iloc[0]['filename']
print(first_file)
print(extract_cycle_features(path, first_file))

00001.csv
{'mean_voltage': 3.4752664759657903, 'max_voltage': 4.246764125510136, 'min_voltage': 2.4706118669327277, 'mean_current': -0.9527668403077523, 'max_current': 0.000252388610586, 'mean_temperature': 8.272422519178683, 'max_temperature': 12.376815880696888, 'discharge_time': 6436.141}
